# Camada Bronze — `acessos`

**Disciplina:** Mineração de Dados — FATEC Matão-SP (Prof. Rodrigo Vidrich)  
**Projeto:** Trabalho Final (P2) — Arquitetura Medallion (`Bronze → Silver → Gold → Power BI`)  
**Integrantes:** Jerônimo Barbieri Junior, Luan Chierato Oliveira, Miguel Miranda Morandini, Victor Hugo Malipense Testi  
**Data:** 13/11/2026  

**Descrição da tabela:** Logs de acessos ao ambiente virtual (tempo logado e páginas)  
**Arquivo bruto:** `Atividade11 - acessos.txt`

In [1]:
import json
import sys
from pathlib import Path

# Garante que notebooks/ esteja no sys.path independente do diretório de execução
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "notebooks" / "utils").exists():
        if str(candidate / "notebooks") not in sys.path:
            sys.path.insert(0, str(candidate / "notebooks"))
        break

from utils.bronze_utils import (
    get_project_paths,
    ingest_raw_file,
    profile_bronze_table,
    save_bronze_outputs,
)

ROOT_DIR, RAW_DIR, BRONZE_DIR = get_project_paths()
print(f"Origem: {RAW_DIR / 'Atividade11 - acessos.txt'}")
print(f"Destino Bronze: {BRONZE_DIR}")

Origem: C:\Projects\trabalho-final-mineacao-de-dados\Bases TRabalho Final\Atividade11 - acessos.txt
Destino Bronze: C:\Projects\trabalho-final-mineacao-de-dados\data\bronze


In [2]:
df, meta = ingest_raw_file(
    dataset_name='acessos',
    filename='Atividade11 - acessos.txt',
    sep='|',
    mixed_sep=False,
    raw_dir=RAW_DIR,
)

profile = profile_bronze_table(df, meta)
parquet_path, profile_path = save_bronze_outputs(df, profile, 'acessos', BRONZE_DIR)

print(f"Linhas ingeridas: {meta['rows']:,} | Tamanho bruto: {meta['size_mb']} MB")
print(f"IDs únicos de aluno: {profile['unique_id_aluno']:,} | Duplicatas exatas: {profile['exact_duplicates']:,}")
print(f"Salvo em: {parquet_path.name} e {profile_path.name}")
df.head(5)

Linhas ingeridas: 400,000 | Tamanho bruto: 9.2981 MB
IDs únicos de aluno: 275,613 | Duplicatas exatas: 0
Salvo em: acessos.parquet e acessos_profile.json


,id_aluno,data,tempo_logado,paginas,_bronze_source_file,_bronze_ingested_at
0,487856,2023-02-12,404,34,Atividade11 - acessos.txt,2026-11-13T18:55:09Z
1,142068,2024-03-07,168,44,Atividade11 - acessos.txt,2026-11-13T18:55:09Z
2,440273,2023-10-01,334,22,Atividade11 - acessos.txt,2026-11-13T18:55:09Z
3,138322,2023-06-10,201,13,Atividade11 - acessos.txt,2026-11-13T18:55:09Z
4,165278,2022-04-18,103,1,Atividade11 - acessos.txt,2026-11-13T18:55:09Z


In [3]:
print("=== Campos vazios por coluna ===")
print(json.dumps(profile["empty_counts"], ensure_ascii=False, indent=2))
if profile["anomalies"]:
    print("\n=== Anomalias detectadas para tratamento na Silver ===")
    print(json.dumps(profile["anomalies"], ensure_ascii=False, indent=2))

=== Campos vazios por coluna ===
{
  "id_aluno": 0,
  "data": 0,
  "tempo_logado": 0,
  "paginas": 0
}
